In [27]:
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain_groq import ChatGroq
from langchain.agents import create_agent



In [28]:
load_dotenv()

True

In [29]:
#Load Data
pdf_loader = PyPDFLoader('../../data/medical_report.pdf')
docs =pdf_loader.load()

In [30]:
#Chunks
doc_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
split_data = doc_splitter.split_documents(docs)

In [31]:
#Embeddings
try:
    embeddings  # Check if variable already exists
    print("Embeddings already loaded. Skipping...")
except NameError:
    print("Loading embeddings for the first time...")
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-mpnet-base-v2"
    )

Embeddings already loaded. Skipping...


In [32]:
#Vector store
vector_store = InMemoryVectorStore.from_documents(
    documents=split_data,
    embedding=embeddings
)

In [33]:
# same_record = vector_store.similarity_search("Patient Name")
# same_record[0].page_content

In [34]:
### agent - tools , llm , prompt

In [35]:
@tool
def retriever_tool(user_query:str):
    """
        This tool will help you to retrieve relevant data of PDF docs,and the documents have details about medical reports.
    """
    relevant_data = vector_store.similarity_search(query=user_query,k=4)
    #print(relevant_data)
    print("Tool Called: ", query)
    context = '\n\n---\n\n'.join([doc.page_content for doc in relevant_data])
    return context

In [36]:
#print(retriever_tool.invoke("Patient Name"))


In [37]:

llm_model = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    streaming=True
    # other params...
)

In [38]:
System_Prompt = """
    You are a helpful medical assistant.
    You MUST use the `retriever_tool` for ANY question about the patient,
    doctors, diagnoses, or medical information in the PDF.
    Never answer from memory — always retrieve first.
"""

In [39]:
agent = create_agent(
    model=llm_model,
    tools=[retriever_tool],
    system_prompt=System_Prompt
                     )

In [40]:
query = "What is the patient's full name and who are the attending doctors?"
response = agent.invoke({"messages":[{"role":"user", "content":query}]})

Tool Called:  What is the patient's full name and who are the attending doctors?
Tool Called:  What is the patient's full name and who are the attending doctors?
Tool Called:  What is the patient's full name and who are the attending doctors?


In [41]:
result = response["messages"][-1].content

In [42]:
print(result)

Based on the medical report retrieved, here is the information available:

**Patient's Full Name:** The patient's name is **not explicitly listed** in the retrieved portions of the report. The document appears to be a Dr. Lal PathLabs diagnostic report, but the patient identification section (typically at the top of page 1) was not captured in the retrieval results.

**Attending/Validating Doctors:**
- **Dr. Kiran Bhargava Pathak** – MD, Pathology, Chief of Laboratory, Dr. Lal PathLabs Ltd

This is the only doctor named in the report. The report is a computer-generated diagnostic report validated by an Authorized Medical Practitioner/Doctor from Dr. Lal PathLabs Ltd.

If you need the patient's full name, it may be located in a section of the report that wasn't fully retrieved. Would you like me to search for additional details?
